In [0]:
customer_df = spark.read.table('de_mini_project.azure_blob_storage.customer')
inventory_df = spark.read.table('de_mini_project.azure_blob_storage.inventory')
product_df = spark.read.table('de_mini_project.azure_blob_storage.product')
sales_df = spark.read.table('de_mini_project.azure_blob_storage.sales')
stores_df = spark.read.table('de_mini_project.azure_blob_storage.stores')
transaction_df = spark.read.table('de_mini_project.azure_blob_storage.transaction')

In [0]:
display(customer_df)
display(inventory_df)
display(product_df)
display(sales_df)
display(stores_df)
display(transaction_df)

In [0]:
from pyspark.sql import functions as F
customer_df = customer_df.drop("_file","_line","_modified","_fivetran_synced")

customer_df = customer_df.toDF(
    "customer_id",
    "full_name",
    "contact_info",
    "joined_date",
    "gender_code"
)
customer_df = customer_df.withColumn("joined_date",F.coalesce(
        F.try_to_date(F.col("joined_date"), "dd-MM-yyyy"),
        F.try_to_date(F.col("joined_date"), "yyyy.MM.dd"),
        F.try_to_date(F.col("joined_date"), "MMM dd, yyyy"),
        F.try_to_date(F.col("joined_date"), "yyyy-MM-dd")
    )
)
customer_df = customer_df.filter(customer_df.contact_info != "NULL")
customer_df = customer_df.withColumn("full_name",F.regexp_replace("full_name",'"',''))
customer_df = customer_df.withColumn("full_name",F.regexp_replace("full_name",'_',' '))

display(customer_df)

In [0]:
inventory_df = inventory_df.drop("_file","_line","_modified","_fivetran_synced")
inventory_df = inventory_df.toDF(
    "sku_id",
    "st_id",
    "stock_on_hand",
    "last_audit_date"
)
inventory_df = inventory_df.withColumn("last_audit_date",F.coalesce(
        F.try_to_date(F.col("last_audit_date"), "yyyyMMdd")
    )
)
inventory_df = inventory_df.withColumnRenamed("st_id","store_id")
display(inventory_df)

In [0]:
sales_df = sales_df.drop("_file","_line","_modified","_fivetran_synced")
sales_df = sales_df.withColumnRenamed("Trxn_ID#","transaction_id").withColumnRenamed("PROD_CODE_ID","sku_id").withColumnRenamed("Cust_ID_99","customer_id").withColumnRenamed("Store_Loc_ID","store_id").withColumnRenamed("Qty_Sold","qty_sold").withColumnRenamed("_Unit_Price_","unit_price").withColumnRenamed("!Date_Ref!","date")
sales_df = sales_df.withColumn("date",F.coalesce(
    F.try_to_date(F.col("date"), "dd-MM-yyyy"),
    F.try_to_date(F.col("date"), "MM-dd-yyyy"),
    F.try_to_date(F.col("date"), "dd-MMM-yy"),
    F.try_to_date(F.col("date"), "yyyy.MM.dd"),
    F.try_to_date(F.col("date"), "MMM dd, yyyy"),
    F.try_to_date(F.col("date"), "yyyyMMdd"),

))
sales_df = sales_df.filter(sales_df.customer_id != "NULL")
display(sales_df)